# Day 09. Exercise 00
# Regularization

## 0. Imports

In [112]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from joblib import dump

## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [113]:
df = pd.read_csv('../data/dayofweek.csv')

In [114]:
df

,uid,labname,numTrials,dayofweek,hour
0,user_4,project1,1,4,5
1,user_4,project1,2,4,5
2,user_4,project1,3,4,5
3,user_4,project1,4,4,5
4,user_4,project1,5,4,5
...,...,...,...,...,...
1681,user_19,laba06s,9,3,20
1682,user_1,laba06s,6,3,20
1683,user_1,laba06s,7,3,20
1684,user_1,laba06s,8,3,20


In [115]:
X = df.drop('dayofweek', axis=1)

In [116]:
y = df['dayofweek']

In [117]:
# кодирование категориальных признаков в X
columns_to_encode = ['uid', 'labname']
encoder = OneHotEncoder(handle_unknown='ignore')
encoded_arr = encoder.fit_transform(X[columns_to_encode]).toarray()

In [118]:
encoded_df = pd.DataFrame(encoded_arr, columns=encoder.get_feature_names_out(columns_to_encode))

In [119]:
X = pd.concat([X, encoded_df], axis=1)
X = X.drop(columns=columns_to_encode)

In [120]:
X

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,9,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1682,6,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1683,7,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1684,8,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [121]:
# масштабирование признаков
scaler = StandardScaler()
X[['numTrials', 'hour']] = scaler.fit_transform(X[['numTrials', 'hour']])

In [122]:
X

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,-0.788667,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.756764,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-0.724861,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,-0.692958,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-0.661055,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,-0.533442,0.945382,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1682,-0.629151,0.945382,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1683,-0.597248,0.945382,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1684,-0.565345,0.945382,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [123]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [124]:
def crossval(n_splits, X, y, model):
    stratified_k_fold = StratifiedKFold(n_splits=n_splits, random_state=21, shuffle=True)
    valid_scores = []
    
    for train_index, valid_index in stratified_k_fold.split(X_train, y_train):
        X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
        y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]
    
        model.fit(X_train_fold, y_train_fold)
        train_pred = model.predict(X_train_fold)
        valid_pred = model.predict(X_valid_fold)
        
        curr_train_score = accuracy_score(y_train_fold, train_pred)
        curr_valid_score = accuracy_score(y_valid_fold, valid_pred)
        
        valid_scores.append(curr_valid_score)
    
        print(f'train -  {curr_train_score:.5f}   |   valid -  {curr_valid_score:.5f}')
    
    print(f'Average accuracy on crossval is {np.mean(valid_scores):.5f}')

In [125]:
%%time
lr = LogisticRegression(random_state=21, fit_intercept=False)
crossval(10, X, y, lr)

train -  0.64056   |   valid -  0.65926
train -  0.63561   |   valid -  0.62222
train -  0.64468   |   valid -  0.60000
train -  0.64056   |   valid -  0.64444
train -  0.65375   |   valid -  0.60741
train -  0.62902   |   valid -  0.60000
train -  0.66117   |   valid -  0.60000
train -  0.63726   |   valid -  0.54074
train -  0.63756   |   valid -  0.66418
train -  0.64745   |   valid -  0.61194
Average accuracy on crossval is 0.61502
CPU times: user 1.9 s, sys: 39.7 ms, total: 1.94 s
Wall time: 2.03 s


### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [126]:
penalties = [None, 'l1', 'l2', 'elasticnet']
for penalty in penalties:
    if penalty == 'elasticnet':
        lr = LogisticRegression(penalty=penalty, solver='saga', l1_ratio=0.5, max_iter=2000, random_state=21, fit_intercept=False)
    else:
        if penalty in ['l1', 'l2']:
            solver = 'liblinear'
        else:
            solver = 'lbfgs'
        lr = LogisticRegression(penalty=penalty, solver=solver, max_iter=2000, random_state=21, fit_intercept=False)

    print(f'Penalty is {penalty}')
    crossval(3, X_train, y_train, lr)
    print()

Penalty is None
train -  0.65479   |   valid -  0.63556
train -  0.67408   |   valid -  0.62584
train -  0.66296   |   valid -  0.63029
Average accuracy on crossval is 0.63056

Penalty is l1
train -  0.59577   |   valid -  0.58000
train -  0.63070   |   valid -  0.58797
train -  0.61513   |   valid -  0.59020
Average accuracy on crossval is 0.58606

Penalty is l2
train -  0.58129   |   valid -  0.55333
train -  0.63181   |   valid -  0.58129
train -  0.60512   |   valid -  0.58575
Average accuracy on crossval is 0.57346

Penalty is elasticnet
train -  0.61024   |   valid -  0.58889
train -  0.65072   |   valid -  0.59243
train -  0.62625   |   valid -  0.60356
Average accuracy on crossval is 0.59496



## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [127]:
%%time
svc = SVC(probability=True, kernel='linear', random_state=21)
stratified_k_fold = StratifiedKFold(n_splits=10, random_state=21, shuffle=True)
valid_scores = []

for train_index, valid_index in stratified_k_fold.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]
    
    svc.fit(X_train_fold, y_train_fold)
    
    train_pred = svc.predict(X_train_fold)
    valid_pred = svc.predict(X_valid_fold)
    
    train_score = accuracy_score(y_train_fold, train_pred)
    valid_score = accuracy_score(y_valid_fold, valid_pred)
    
    valid_scores.append(valid_score)
    
    print(f'train -  {train_score:.5f}   |   valid -  {valid_score:.5f}')
    
average_accuracy = np.mean(valid_scores)
std_accuracy = np.std(valid_scores)

print(f'Average accuracy on crossval is {average_accuracy:.5}')
print(f'Std is {std_accuracy:.5}')


train -  0.70651   |   valid -  0.68148
train -  0.68920   |   valid -  0.64444
train -  0.69744   |   valid -  0.66667
train -  0.68920   |   valid -  0.65926
train -  0.69497   |   valid -  0.63704
train -  0.68673   |   valid -  0.68148
train -  0.69827   |   valid -  0.61481
train -  0.70486   |   valid -  0.57778
train -  0.68863   |   valid -  0.72388
train -  0.71005   |   valid -  0.64179
Average accuracy on crossval is 0.65286
Std is 0.038003
CPU times: user 10.2 s, sys: 308 ms, total: 10.5 s
Wall time: 10.1 s


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [128]:
Cs = [0.1, 1, 10, 100]
# stratified_k_fold = StratifiedKFold(n_splits=10, random_state=21, shuffle=True)

for C in Cs:
    print(f'C = {C}')
    svc = SVC(C=C, probability=True, kernel='linear', random_state=21)
    valid_scores = []

    for train_index, valid_index in stratified_k_fold.split(X_train, y_train):
        X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
        y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]

        svc.fit(X_train_fold, y_train_fold)

        train_pred = svc.predict(X_train_fold)
        valid_pred = svc.predict(X_valid_fold)

        train_score = accuracy_score(y_train_fold, train_pred)
        valid_score = accuracy_score(y_valid_fold, valid_pred)

        valid_scores.append(valid_score)

        print(f'train -  {train_score:.5f}   |   valid -  {valid_score:.5f}')

    average_accuracy = np.mean(valid_scores)
    std_accuracy = np.std(valid_scores)

    print(f'Average accuracy on crossval is {average_accuracy:.5}')
    print(f'Std is {std_accuracy:.5}')
    print()

C = 0.1
train -  0.57049   |   valid -  0.55556
train -  0.56884   |   valid -  0.59259
train -  0.57543   |   valid -  0.54074
train -  0.56142   |   valid -  0.60000
train -  0.59110   |   valid -  0.57037
train -  0.57873   |   valid -  0.53333
train -  0.59687   |   valid -  0.54074
train -  0.59439   |   valid -  0.52593
train -  0.56590   |   valid -  0.58209
train -  0.58731   |   valid -  0.53731
Average accuracy on crossval is 0.55787
Std is 0.02522

C = 1
train -  0.70651   |   valid -  0.68148
train -  0.68920   |   valid -  0.64444
train -  0.69744   |   valid -  0.66667
train -  0.68920   |   valid -  0.65926
train -  0.69497   |   valid -  0.63704
train -  0.68673   |   valid -  0.68148
train -  0.69827   |   valid -  0.61481
train -  0.70486   |   valid -  0.57778
train -  0.68863   |   valid -  0.72388
train -  0.71005   |   valid -  0.64179
Average accuracy on crossval is 0.65286
Std is 0.038003

C = 10
train -  0.76175   |   valid -  0.71852
train -  0.76340   |   val

## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [129]:
dt = DecisionTreeClassifier(max_depth=10, random_state=21)
stratified_k_fold = StratifiedKFold(n_splits=10, random_state=21, shuffle=True)
valid_scores = []

for train_index, valid_index in stratified_k_fold.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]
    
    dt.fit(X_train_fold, y_train_fold)
    
    train_pred = dt.predict(X_train_fold)
    valid_pred = dt.predict(X_valid_fold)
    
    train_score = accuracy_score(y_train_fold, train_pred)
    valid_score = accuracy_score(y_valid_fold, valid_pred)
    
    valid_scores.append(valid_score)
    
    print(f'train -  {train_score:.5f}   |   valid -  {valid_score:.5f}')

        
average_accuracy = np.mean(valid_scores)
std_accuracy = np.std(valid_scores)

print(f'Average accuracy on crossval is {average_accuracy:.5}')
print(f'Std is {std_accuracy:.5}')

train -  0.80874   |   valid -  0.77037
train -  0.79802   |   valid -  0.70370
train -  0.81286   |   valid -  0.72593
train -  0.80049   |   valid -  0.74815
train -  0.80956   |   valid -  0.68889
train -  0.78978   |   valid -  0.74074
train -  0.80627   |   valid -  0.60741
train -  0.82688   |   valid -  0.71111
train -  0.78995   |   valid -  0.79104
train -  0.80313   |   valid -  0.70896
Average accuracy on crossval is 0.71963
Std is 0.047909


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [130]:
max_depth = (2, 5, 10, 15, 20, 50, 100)

for depth in max_depth:
    print(f'max_depth = {depth}')
    valid_scores = []
    
    dt = DecisionTreeClassifier(max_depth=depth, random_state=21)

    for train_index, valid_index in stratified_k_fold.split(X_train, y_train):
        X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
        y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]

        dt.fit(X_train_fold, y_train_fold)

        train_pred = dt.predict(X_train_fold)
        valid_pred = dt.predict(X_valid_fold)

        train_score = accuracy_score(y_train_fold, train_pred)
        valid_score = accuracy_score(y_valid_fold, valid_pred)

        valid_scores.append(valid_score)

        print(f'train -  {train_score:.5f}   |   valid -  {valid_score:.5f}')


    average_accuracy = np.mean(valid_scores)
    std_accuracy = np.std(valid_scores)

    print(f'Average accuracy on crossval is {average_accuracy:.5}')
    print(f'Std is {std_accuracy:.5}')
    print()

max_depth = 2
train -  0.43199   |   valid -  0.45926
train -  0.43116   |   valid -  0.46667
train -  0.43281   |   valid -  0.45185
train -  0.43281   |   valid -  0.45185
train -  0.43611   |   valid -  0.42222
train -  0.43776   |   valid -  0.40741
train -  0.43446   |   valid -  0.43704
train -  0.43364   |   valid -  0.44444
train -  0.43740   |   valid -  0.41045
train -  0.43904   |   valid -  0.39552
Average accuracy on crossval is 0.43467
Std is 0.023103

max_depth = 5
train -  0.58285   |   valid -  0.60741
train -  0.57626   |   valid -  0.52593
train -  0.61253   |   valid -  0.60000
train -  0.58862   |   valid -  0.58519
train -  0.58615   |   valid -  0.51111
train -  0.56224   |   valid -  0.53333
train -  0.58120   |   valid -  0.51852
train -  0.62407   |   valid -  0.51111
train -  0.57414   |   valid -  0.56716
train -  0.56672   |   valid -  0.48507
Average accuracy on crossval is 0.54448
Std is 0.04014

max_depth = 10
train -  0.80874   |   valid -  0.77037
trai

## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [131]:
rf = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)
valid_scores = []

for train_index, valid_index in stratified_k_fold.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]
    
    rf.fit(X_train_fold, y_train_fold)
    
    train_pred = rf.predict(X_train_fold)
    valid_pred = rf.predict(X_valid_fold)
    
    train_score = accuracy_score(y_train_fold, train_pred)
    valid_score = accuracy_score(y_valid_fold, valid_pred)
    
    valid_scores.append(valid_score)
    
    print(f'train -  {train_score:.5f}   |   valid -  {valid_score:.5f}')


average_accuracy = np.mean(valid_scores)
std_accuracy = np.std(valid_scores)

print(f'Average accuracy on crossval is {average_accuracy:.5}')
print(f'Std is {std_accuracy:.5}')

train -  0.97939   |   valid -  0.85185
train -  0.96620   |   valid -  0.85926
train -  0.96208   |   valid -  0.91852
train -  0.97115   |   valid -  0.91852
train -  0.97197   |   valid -  0.88148
train -  0.96538   |   valid -  0.86667
train -  0.96455   |   valid -  0.88889
train -  0.96867   |   valid -  0.87407
train -  0.96458   |   valid -  0.93284
train -  0.96787   |   valid -  0.86567
Average accuracy on crossval is 0.88578
Std is 0.026734


### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [132]:
for max_depth in (10, 50, 100):
    for n_estimators in (100, 200, 300):
        
        print(f'max_depth = {max_depth}')
        print(f'n_estimators = {n_estimators}')
        
        rf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=21)

        valid_scores = []

        for train_index, valid_index in stratified_k_fold.split(X_train, y_train):
            X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
            y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]

            rf.fit(X_train_fold, y_train_fold)

            train_pred = rf.predict(X_train_fold)
            valid_pred = rf.predict(X_valid_fold)

            train_score = accuracy_score(y_train_fold, train_pred)
            valid_score = accuracy_score(y_valid_fold, valid_pred)

            valid_scores.append(valid_score)

            print(f'train -  {train_score:.5f}   |   valid -  {valid_score:.5f}')


        average_accuracy = np.mean(valid_scores)
        std_accuracy = np.std(valid_scores)

        print(f'Average accuracy on crossval is {average_accuracy:.5}')
        print(f'Std is {std_accuracy:.5}')
        print()

max_depth = 10
n_estimators = 100
train -  0.87552   |   valid -  0.82222
train -  0.88871   |   valid -  0.77037
train -  0.89200   |   valid -  0.82222
train -  0.88293   |   valid -  0.84444
train -  0.87964   |   valid -  0.78519
train -  0.89777   |   valid -  0.84444
train -  0.88293   |   valid -  0.79259
train -  0.87634   |   valid -  0.76296
train -  0.89209   |   valid -  0.85821
train -  0.87809   |   valid -  0.76866
Average accuracy on crossval is 0.80713
Std is 0.033652

max_depth = 10
n_estimators = 200
train -  0.88046   |   valid -  0.82222
train -  0.88211   |   valid -  0.78519
train -  0.88623   |   valid -  0.83704
train -  0.90025   |   valid -  0.84444
train -  0.89118   |   valid -  0.80000
train -  0.89613   |   valid -  0.83704
train -  0.87716   |   valid -  0.78519
train -  0.88376   |   valid -  0.80000
train -  0.89292   |   valid -  0.85821
train -  0.87974   |   valid -  0.75373
Average accuracy on crossval is 0.81231
Std is 0.031091

max_depth = 10
n_e

In [133]:
for min_samples_leaf in (2, 5, 10):

    print(f'max_depth = 100')
    print(f'n_estimators = 300')
    print(f'min_samples_leaf = {min_samples_leaf}')

    rf = RandomForestClassifier(n_estimators=300, max_depth=100, random_state=21)

    valid_scores = []

    for train_index, valid_index in stratified_k_fold.split(X_train, y_train):
        X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
        y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]

        rf.fit(X_train_fold, y_train_fold)

        train_pred = rf.predict(X_train_fold)
        valid_pred = rf.predict(X_valid_fold)

        train_score = accuracy_score(y_train_fold, train_pred)
        valid_score = accuracy_score(y_valid_fold, valid_pred)

        valid_scores.append(valid_score)

        print(f'train -  {train_score:.5f}   |   valid -  {valid_score:.5f}')


    average_accuracy = np.mean(valid_scores)
    std_accuracy = np.std(valid_scores)

    print(f'Average accuracy on crossval is {average_accuracy:.5}')
    print(f'Std is {std_accuracy:.5}')
    print()

max_depth = 100
n_estimators = 300
min_samples_leaf = 2
train -  1.00000   |   valid -  0.91111
train -  1.00000   |   valid -  0.88148
train -  1.00000   |   valid -  0.95556
train -  1.00000   |   valid -  0.92593
train -  1.00000   |   valid -  0.92593
train -  1.00000   |   valid -  0.92593
train -  1.00000   |   valid -  0.95556
train -  1.00000   |   valid -  0.88889
train -  1.00000   |   valid -  0.94776
train -  1.00000   |   valid -  0.88060
Average accuracy on crossval is 0.91987
Std is 0.027363

max_depth = 100
n_estimators = 300
min_samples_leaf = 5
train -  1.00000   |   valid -  0.91111
train -  1.00000   |   valid -  0.88148
train -  1.00000   |   valid -  0.95556
train -  1.00000   |   valid -  0.92593
train -  1.00000   |   valid -  0.92593
train -  1.00000   |   valid -  0.92593
train -  1.00000   |   valid -  0.95556
train -  1.00000   |   valid -  0.88889
train -  1.00000   |   valid -  0.94776
train -  1.00000   |   valid -  0.88060
Average accuracy on crossval is

## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [134]:
rf = RandomForestClassifier(max_depth=50, n_estimators = 100, random_state=21)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
accuracy_score(y_test, pred)

0.9378698224852071

In [135]:
df_pred = pd.DataFrame()

In [136]:
df_pred['prediction'] = pred
df_pred['y_test'] = np.array(y_test)

In [137]:
df_pred['error'] = df_pred['prediction'] != df_pred['y_test']

In [138]:
df_pred.groupby('y_test').mean()['error'].sort_values(ascending=False) * 100

y_test
0    25.925926
4    14.285714
2     6.666667
5     5.555556
1     5.454545
3     2.500000
6     1.408451
Name: error, dtype: float64

Answer: the most errors the model makes for the day number 0 (Monday)

In [139]:
dump(rf, 'my_random_forest_model.joblib')

['my_random_forest_model.joblib']